# Prueba de escalabilidad del pipeline — Danper

## Apartado 3.7 del Proyecto Integrador Big Data

### Objetivo
Evaluar cómo cambia el rendimiento del procesamiento de datos agrocomerciales al incrementar el volumen de información.

### Volúmenes de prueba
- Volumen original (×1).
- Réplica de diez veces el volumen original (×10).
- Réplica de cien veces el volumen original (×100).

### Entorno tecnológico
- Sistema operativo: Debian 12.
- Motor de procesamiento: Apache Spark 3.5.1.
- Almacenamiento: Hadoop HDFS.
- Infraestructura: máquina virtual on-premise.
- Procesadores disponibles: 5.
- Memoria RAM: 8 GiB.

### Métricas
- Número de registros procesados.
- Tiempo de ejecución.
- Utilización de CPU.
- Consumo de memoria.
- Comportamiento del rendimiento al aumentar el volumen.

### Consideraciones
Los datos replicados se utilizarán exclusivamente para evaluar el rendimiento. No se incorporarán a las zonas Silver y Gold oficiales ni se utilizarán para análisis de negocio o entrenamiento de modelos.

In [1]:
import os
import time
import gc

import psutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Danper_Prueba_Escalabilidad")
    .master("local[3]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000")
    .getOrCreate()
)

ruta_silver = "hdfs://localhost:9000/danper/silver/produccion"
df_base = spark.read.parquet(ruta_silver)

print("===== PASO 118: CONFIGURACION =====")
print("Version Spark:", spark.version)
print("CPU disponibles:", psutil.cpu_count())
print("RAM total (GiB):", round(
    psutil.virtual_memory().total / (1024 ** 3), 2
))
print("Registros originales:", df_base.count())
print("Columnas:", len(df_base.columns))
print("Ruta utilizada:", ruta_silver)
print("===== PASO 118 FINALIZADO =====")

26/10/11 17:09:56 WARN Utils: Your hostname, Debian12 resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/11 17:09:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 17:09:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

===== PASO 118: CONFIGURACION =====
Version Spark: 3.5.1
CPU disponibles: 5
RAM total (GiB): 8.02


[Stage 1:>                                                          (0 + 2) / 2]

Registros originales: 65922
Columnas: 13
Ruta utilizada: hdfs://localhost:9000/danper/silver/produccion
===== PASO 118 FINALIZADO =====


## Paso 119. Planificación de los volúmenes de prueba

Se utilizará el conjunto Silver de producción agrícola como primera fuente para comprobar el crecimiento de los datos.

Se evaluarán tres escenarios:

- Escenario ×1: volumen original.
- Escenario ×10: diez veces el volumen original.
- Escenario ×100: cien veces el volumen original.

Antes de generar las réplicas, se verificará la cantidad esperada de registros y los recursos disponibles.

Esta comprobación es preparatoria. La prueba definitiva del apartado 3.7 deberá considerar las transformaciones del pipeline y las fuentes correspondientes.

In [2]:
print("===== PASO 119: PLANIFICAR ESCALABILIDAD =====")

registros_originales = df_base.count()

print("\n===== VOLUMENES PREVISTOS =====")
for factor in [1, 10, 100]:
    print(
        f"Escenario x{factor}: "
        f"{registros_originales * factor:,} registros"
    )

print("\n===== RECURSOS DISPONIBLES =====")
memoria = psutil.virtual_memory()
disco = psutil.disk_usage("/")

print(
    "RAM disponible (GiB):",
    round(memoria.available / (1024 ** 3), 2)
)

print(
    "Disco disponible (GiB):",
    round(disco.free / (1024 ** 3), 2)
)

print("\n===== PASO 119 FINALIZADO =====")

===== PASO 119: PLANIFICAR ESCALABILIDAD =====

===== VOLUMENES PREVISTOS =====
Escenario x1: 65,922 registros
Escenario x10: 659,220 registros
Escenario x100: 6,592,200 registros

===== RECURSOS DISPONIBLES =====
RAM disponible (GiB): 4.03
Disco disponible (GiB): 4.78

===== PASO 119 FINALIZADO =====


## Paso 120. Prueba de escalabilidad con volumen original (×1)

Se ejecuta una transformación analítica sobre los 65 922 registros originales del conjunto Silver de producción agrícola.

La operación agrupa los datos por cultivo, año y variable, calculando la suma de los valores y el número de registros de cada grupo.

Se registrará el tiempo de ejecución y el consumo de recursos del proceso Spark.

Esta prueba constituye una línea base técnica de la transformación seleccionada. No representa todavía una ejecución completa del pipeline Bronze–Silver–Gold.

In [3]:
import time
import psutil
from pyspark.sql import functions as F

print("===== PASO 120: PRUEBA BASE X1 =====")

# Crear la transformación analítica
def transformar_produccion(datos):
    return (
        datos
        .groupBy("cultivo", "anio", "variable")
        .agg(
            F.sum("valor").alias("valor_total"),
            F.count("*").alias("cantidad_registros")
        )
    )

# Identificar el proceso Java que ejecuta Spark
java_spark = [
    p for p in psutil.process_iter(["pid", "name", "cmdline"])
    if "org.apache.spark.deploy.SparkSubmit"
    in " ".join(p.info["cmdline"] or [])
]

proceso_spark = (
    psutil.Process(java_spark[0].pid)
    if java_spark else None
)

# Medir recursos antes de ejecutar
ram_antes = psutil.virtual_memory().available
inicio = time.perf_counter()

# Ejecutar la transformación y materializar el resultado
resultado_x1 = transformar_produccion(df_base)
grupos_x1 = resultado_x1.collect()

tiempo_x1 = time.perf_counter() - inicio
ram_despues = psutil.virtual_memory().available

# Guardar los resultados
registros_x1 = registros_originales
cantidad_grupos_x1 = len(grupos_x1)

print("\n===== RESULTADOS X1 =====")
print("Registros procesados:", registros_x1)
print("Grupos obtenidos:", cantidad_grupos_x1)
print(f"Tiempo de ejecución: {tiempo_x1:.4f} segundos")

print("\n===== RECURSOS =====")
print(
    "RAM disponible antes (GiB):",
    round(ram_antes / (1024**3), 2)
)
print(
    "RAM disponible después (GiB):",
    round(ram_despues / (1024**3), 2)
)

if proceso_spark:
    print(
        "RAM residente del proceso Spark (MiB):",
        round(proceso_spark.memory_info().rss / (1024**2), 2)
    )

print("\n===== PASO 120 FINALIZADO =====")

===== PASO 120: PRUEBA BASE X1 =====


[Stage 7:>                                                          (0 + 2) / 2]


===== RESULTADOS X1 =====
Registros procesados: 65922
Grupos obtenidos: 834
Tiempo de ejecución: 1.9945 segundos

===== RECURSOS =====
RAM disponible antes (GiB): 4.0
RAM disponible después (GiB): 3.93
RAM residente del proceso Spark (MiB): 464.34

===== PASO 120 FINALIZADO =====


## Paso 121. Prueba de escalabilidad ×10

Se replica temporalmente el conjunto Silver de producción agrícola hasta alcanzar diez veces su volumen original.

La réplica se utiliza exclusivamente para medir el rendimiento. Se agrega una columna técnica denominada `id_replica` para diferenciar las copias, sin alterar los datos agrícolas originales.

Posteriormente, se ejecuta la misma agregación por cultivo, año y variable utilizada en la prueba ×1.

Se registran la cantidad de registros, el tiempo de ejecución y una observación del consumo de memoria.

Los datos replicados no se almacenan en las zonas oficiales Bronze, Silver o Gold.

In [4]:
import time
import psutil
from pyspark.sql import functions as F

print("===== PASO 121: PRUEBA ESCALABILIDAD X10 =====")

# Generar diez identificadores de réplica
replicas_x10 = spark.range(10).select(
    F.col("id").alias("id_replica")
)

# Replicar los datos de forma temporal
df_x10 = df_base.crossJoin(F.broadcast(replicas_x10))

# Cantidad esperada de registros
registros_esperados_x10 = registros_originales * 10

print("Registros esperados:", registros_esperados_x10)

# Ejecutar la misma agregación utilizada en X1
ram_antes_x10 = psutil.virtual_memory().available
inicio_x10 = time.perf_counter()

resultado_x10 = transformar_produccion(df_x10)
grupos_x10 = resultado_x10.collect()

tiempo_x10 = time.perf_counter() - inicio_x10
ram_despues_x10 = psutil.virtual_memory().available

# Registrar resultados
cantidad_grupos_x10 = len(grupos_x10)

print("\n===== RESULTADOS X10 =====")
print("Registros previstos:", registros_esperados_x10)
print("Grupos obtenidos:", cantidad_grupos_x10)
print(f"Tiempo de ejecución: {tiempo_x10:.4f} segundos")

print("\n===== RECURSOS =====")
print(
    "RAM disponible antes (GiB):",
    round(ram_antes_x10 / (1024**3), 2)
)
print(
    "RAM disponible después (GiB):",
    round(ram_despues_x10 / (1024**3), 2)
)

if proceso_spark:
    print(
        "RAM residente de Spark (MiB):",
        round(proceso_spark.memory_info().rss / (1024**2), 2)
    )

print("\n===== COMPARACION CON X1 =====")
print(f"Tiempo X1: {tiempo_x1:.4f} segundos")
print(f"Tiempo X10: {tiempo_x10:.4f} segundos")
print(f"Factor de crecimiento del tiempo: {tiempo_x10 / tiempo_x1:.2f} veces")

print("\n===== PASO 121 FINALIZADO =====")

===== PASO 121: PRUEBA ESCALABILIDAD X10 =====
Registros esperados: 659220


[Stage 11:>                                                         (0 + 2) / 2]


===== RESULTADOS X10 =====
Registros previstos: 659220
Grupos obtenidos: 834
Tiempo de ejecución: 1.8217 segundos

===== RECURSOS =====
RAM disponible antes (GiB): 3.87
RAM disponible después (GiB): 3.76
RAM residente de Spark (MiB): 651.41

===== COMPARACION CON X1 =====
Tiempo X1: 1.9945 segundos
Tiempo X10: 1.8217 segundos
Factor de crecimiento del tiempo: 0.91 veces

===== PASO 121 FINALIZADO =====


## Paso 122. Validación de la réplica ×10

Se comprueba que la réplica contiene exactamente diez veces los registros originales y que cada identificador de réplica corresponde a una copia completa del conjunto de producción.

Esta validación permite confirmar que el volumen de entrada es el esperado antes de continuar con el escenario ×100.

In [5]:
print("===== PASO 122: VALIDAR REPLICA X10 =====")

# Contar los registros efectivos de la réplica
registros_reales_x10 = df_x10.count()

print("\n===== CANTIDAD DE REGISTROS =====")
print("Registros originales:", registros_originales)
print("Registros esperados X10:", registros_esperados_x10)
print("Registros reales X10:", registros_reales_x10)

print("\n===== VALIDACION =====")
replica_correcta_x10 = (
    registros_reales_x10 == registros_esperados_x10
)

print("Cantidad correcta:", replica_correcta_x10)

print("\n===== DISTRIBUCION POR REPLICA =====")
df_x10.groupBy("id_replica").count().orderBy("id_replica").show(
    10, truncate=False
)

print("\n===== PASO 122 FINALIZADO =====")

===== PASO 122: VALIDAR REPLICA X10 =====

===== CANTIDAD DE REGISTROS =====
Registros originales: 65922
Registros esperados X10: 659220
Registros reales X10: 659220

===== VALIDACION =====
Cantidad correcta: True

===== DISTRIBUCION POR REPLICA =====
+----------+-----+
|id_replica|count|
+----------+-----+
|0         |65922|
|1         |65922|
|2         |65922|
|3         |65922|
|4         |65922|
|5         |65922|
|6         |65922|
|7         |65922|
|8         |65922|
|9         |65922|
+----------+-----+


===== PASO 122 FINALIZADO =====


## Paso 123. Prueba de escalabilidad ×100

Se realiza una réplica temporal de cien veces el volumen original del conjunto Silver de producción agrícola.

La réplica se genera mediante una operación Cross Join con cien identificadores técnicos.

Se ejecuta la misma transformación analítica utilizada en las pruebas ×1 y ×10, agrupando por cultivo, año y variable.

Se registran el tiempo de ejecución, el número de grupos obtenidos y el consumo de memoria observado.

Los datos replicados se utilizan exclusivamente para evaluar rendimiento y no se escriben en las zonas oficiales del Data Lake.

In [6]:
import time
import psutil
from pyspark.sql import functions as F

print("===== PASO 123: PRUEBA ESCALABILIDAD X100 =====")

# Crear 100 identificadores técnicos de réplica
replicas_x100 = spark.range(100).select(
    F.col("id").alias("id_replica")
)

# Construir la réplica temporal
df_x100 = df_base.crossJoin(
    F.broadcast(replicas_x100)
)

registros_esperados_x100 = registros_originales * 100

print("Registros previstos:", registros_esperados_x100)

# Registrar recursos antes de ejecutar
ram_antes_x100 = psutil.virtual_memory().available

inicio_x100 = time.perf_counter()

# Ejecutar exactamente la misma transformación
resultado_x100 = transformar_produccion(df_x100)

# Materializar el resultado para forzar la ejecución
grupos_x100 = resultado_x100.collect()

tiempo_x100 = time.perf_counter() - inicio_x100

ram_despues_x100 = psutil.virtual_memory().available

cantidad_grupos_x100 = len(grupos_x100)

print("\n===== RESULTADOS X100 =====")
print("Registros previstos:", registros_esperados_x100)
print("Grupos obtenidos:", cantidad_grupos_x100)
print(f"Tiempo de ejecución: {tiempo_x100:.4f} segundos")

print("\n===== RECURSOS =====")
print(
    "RAM disponible antes (GiB):",
    round(ram_antes_x100 / (1024**3), 2)
)
print(
    "RAM disponible después (GiB):",
    round(ram_despues_x100 / (1024**3), 2)
)

if proceso_spark:
    print(
        "RAM residente de Spark (MiB):",
        round(proceso_spark.memory_info().rss / (1024**2), 2)
    )

print("\n===== COMPARACION =====")
print(f"Tiempo X1: {tiempo_x1:.4f} segundos")
print(f"Tiempo X10: {tiempo_x10:.4f} segundos")
print(f"Tiempo X100: {tiempo_x100:.4f} segundos")

print(
    "Crecimiento X100 respecto a X1:",
    round(tiempo_x100 / tiempo_x1, 2),
    "veces"
)

print("\n===== PASO 123 FINALIZADO =====")

===== PASO 123: PRUEBA ESCALABILIDAD X100 =====
Registros previstos: 6592200


[Stage 23:=============================>                            (1 + 1) / 2]


===== RESULTADOS X100 =====
Registros previstos: 6592200
Grupos obtenidos: 834
Tiempo de ejecución: 2.8952 segundos

===== RECURSOS =====
RAM disponible antes (GiB): 3.59
RAM disponible después (GiB): 3.38
RAM residente de Spark (MiB): 866.35

===== COMPARACION =====
Tiempo X1: 1.9945 segundos
Tiempo X10: 1.8217 segundos
Tiempo X100: 2.8952 segundos
Crecimiento X100 respecto a X1: 1.45 veces

===== PASO 123 FINALIZADO =====


## Paso 124. Validación de la réplica ×100

Se verifica que la réplica de escalabilidad contiene exactamente cien veces el número de registros originales del conjunto Silver de producción agrícola.

También se comprueba que existen cien identificadores de réplica y que cada copia conserva los 65 922 registros originales.

Esta validación se realiza exclusivamente con fines de evaluación técnica.

In [7]:
from pyspark.sql import functions as F

print("===== PASO 124: VALIDAR REPLICA X100 =====")

# Contar registros efectivos
registros_reales_x100 = df_x100.count()

print("\n===== CANTIDAD DE REGISTROS =====")
print("Registros originales:", registros_originales)
print("Registros esperados X100:", registros_esperados_x100)
print("Registros reales X100:", registros_reales_x100)

# Comprobar la cantidad de copias
distribucion_x100 = (
    df_x100
    .groupBy("id_replica")
    .agg(F.count("*").alias("registros"))
)

resumen_x100 = distribucion_x100.agg(
    F.count("*").alias("numero_replicas"),
    F.min("registros").alias("minimo_por_replica"),
    F.max("registros").alias("maximo_por_replica")
).first()

validacion_x100 = (
    registros_reales_x100 == registros_esperados_x100
    and resumen_x100["numero_replicas"] == 100
    and resumen_x100["minimo_por_replica"] == registros_originales
    and resumen_x100["maximo_por_replica"] == registros_originales
)

print("\n===== VALIDACION =====")
print("Número de réplicas:", resumen_x100["numero_replicas"])
print("Registros mínimos por réplica:", resumen_x100["minimo_por_replica"])
print("Registros máximos por réplica:", resumen_x100["maximo_por_replica"])
print("Validación correcta:", validacion_x100)

print("\n===== PASO 124 FINALIZADO =====")

===== PASO 124: VALIDAR REPLICA X100 =====

===== CANTIDAD DE REGISTROS =====
Registros originales: 65922
Registros esperados X100: 6592200
Registros reales X100: 6592200

===== VALIDACION =====
Número de réplicas: 100
Registros mínimos por réplica: 65922
Registros máximos por réplica: 65922
Validación correcta: True

===== PASO 124 FINALIZADO =====


## Paso 125. Verificación del procesamiento de las réplicas

Se incorpora el identificador técnico de réplica a la agrupación para comprobar que Spark distingue las cien copias del conjunto original.

Se espera obtener 834 grupos por réplica, equivalentes a 83 400 grupos para el escenario ×100.

Esta prueba es una comprobación adicional del experimento de escalabilidad, no una ejecución completa del pipeline Bronze–Silver–Gold.

In [8]:
import time
from pyspark.sql import functions as F

print("===== PASO 125: VERIFICAR PROCESAMIENTO X100 =====")

inicio = time.perf_counter()

# Agrupar cada copia por separado
resultado_replicas = (
    df_x100
    .groupBy("id_replica", "cultivo", "anio", "variable")
    .agg(
        F.sum("valor").alias("valor_total"),
        F.count("*").alias("cantidad_registros")
    )
)

# Ejecutar la agregación y recuperar los resultados
filas = resultado_replicas.collect()

tiempo_verificacion = time.perf_counter() - inicio

grupos_reales = len(filas)
grupos_esperados = cantidad_grupos_x1 * 100

# Comprobar grupos por réplica
conteo_grupos = {}

for fila in filas:
    identificador = fila["id_replica"]
    conteo_grupos[identificador] = (
        conteo_grupos.get(identificador, 0) + 1
    )

validacion = (
    grupos_reales == grupos_esperados
    and len(conteo_grupos) == 100
    and all(
        cantidad == cantidad_grupos_x1
        for cantidad in conteo_grupos.values()
    )
)

print("\n===== RESULTADOS =====")
print("Réplicas procesadas:", len(conteo_grupos))
print("Grupos esperados:", grupos_esperados)
print("Grupos obtenidos:", grupos_reales)
print(f"Tiempo de ejecución: {tiempo_verificacion:.4f} segundos")
print("Validación correcta:", validacion)

print("\n===== PASO 125 FINALIZADO =====")

===== PASO 125: VERIFICAR PROCESAMIENTO X100 =====



===== RESULTADOS =====
Réplicas procesadas: 100
Grupos esperados: 83400
Grupos obtenidos: 83400
Tiempo de ejecución: 2.8014 segundos
Validación correcta: True

===== PASO 125 FINALIZADO =====


## Paso 137. Preparación de las fuentes experimentales

Se preparan las siete fuentes Silver utilizadas por el procesamiento Gold para los escenarios ×1, ×10 y ×100.

La replicación conserva el esquema y los valores originales. Los registros duplicados se utilizan exclusivamente para medir rendimiento, no para obtener conclusiones de negocio.

Los conjuntos experimentales se almacenan bajo `/danper/pruebas_escalabilidad`, sin modificar los directorios oficiales del Data Lake.

In [9]:
from pyspark.sql import functions as F
import time

print("===== PASO 137: PREPARAR FUENTES EXPERIMENTALES =====")

raiz = "hdfs://localhost:9000/danper/pruebas_escalabilidad"

fuentes = [
    "produccion",
    "exportaciones/regionales",
    "exportaciones/danper",
    "agroindustria",
    "costos",
    "abastecimiento",
    "mercado_productores"
]

factores = [1, 10, 100]

# Acceso al sistema de archivos de HDFS
hconf = spark.sparkContext._jsc.hadoopConfiguration()
Path = spark._jvm.org.apache.hadoop.fs.Path

def existe_hdfs(ruta):
    p = Path(ruta)
    return p.getFileSystem(hconf).exists(p)

resumen_fuentes = []

for factor in factores:

    print(f"\n===== ESCENARIO X{factor} =====")

    for fuente in fuentes:

        origen = f"hdfs://localhost:9000/danper/silver/{fuente}"
        destino = f"{raiz}/silver_x{factor}/{fuente}"

        if existe_hdfs(destino):
            print(f"[YA EXISTE] {destino}")
            continue

        inicio = time.perf_counter()

        datos = spark.read.parquet(origen)

        if factor == 1:
            replica = datos
        else:
            ids = spark.range(factor).select(
                F.col("id").alias("_id_replica")
            )

            replica = (
                datos.crossJoin(F.broadcast(ids))
                .drop("_id_replica")
            )

        # Máximo 4 particiones para controlar archivos.
        replica.coalesce(4).write.mode(
            "errorifexists"
        ).parquet(destino)

        duracion = time.perf_counter() - inicio

        resumen_fuentes.append({
            "escenario": factor,
            "fuente": fuente,
            "tiempo_segundos": round(duracion, 2)
        })

        print(
            f"[CREADO] {fuente} "
            f"| escenario x{factor} "
            f"| tiempo: {duracion:.2f} s"
        )

print("\n===== RESUMEN =====")

for registro in resumen_fuentes:
    print(registro)

print("\n===== PASO 137 FINALIZADO =====")

===== PASO 137: PREPARAR FUENTES EXPERIMENTALES =====

===== ESCENARIO X1 =====


[CREADO] produccion | escenario x1 | tiempo: 3.57 s
[CREADO] exportaciones/regionales | escenario x1 | tiempo: 0.71 s
[CREADO] exportaciones/danper | escenario x1 | tiempo: 0.63 s
[CREADO] agroindustria | escenario x1 | tiempo: 0.63 s
[CREADO] costos | escenario x1 | tiempo: 0.54 s
[CREADO] abastecimiento | escenario x1 | tiempo: 0.55 s
[CREADO] mercado_productores | escenario x1 | tiempo: 0.76 s

===== ESCENARIO X10 =====


[CREADO] produccion | escenario x10 | tiempo: 1.39 s


[CREADO] exportaciones/regionales | escenario x10 | tiempo: 0.98 s
[CREADO] exportaciones/danper | escenario x10 | tiempo: 0.59 s
[CREADO] agroindustria | escenario x10 | tiempo: 0.55 s
[CREADO] costos | escenario x10 | tiempo: 0.94 s
[CREADO] abastecimiento | escenario x10 | tiempo: 0.52 s
[CREADO] mercado_productores | escenario x10 | tiempo: 0.50 s

===== ESCENARIO X100 =====


[CREADO] produccion | escenario x100 | tiempo: 6.61 s
[CREADO] exportaciones/regionales | escenario x100 | tiempo: 0.63 s
[CREADO] exportaciones/danper | escenario x100 | tiempo: 0.73 s
[CREADO] agroindustria | escenario x100 | tiempo: 0.78 s


[CREADO] costos | escenario x100 | tiempo: 0.89 s
[CREADO] abastecimiento | escenario x100 | tiempo: 0.48 s
[CREADO] mercado_productores | escenario x100 | tiempo: 0.60 s

===== RESUMEN =====
{'escenario': 1, 'fuente': 'produccion', 'tiempo_segundos': 3.57}
{'escenario': 1, 'fuente': 'exportaciones/regionales', 'tiempo_segundos': 0.71}
{'escenario': 1, 'fuente': 'exportaciones/danper', 'tiempo_segundos': 0.63}
{'escenario': 1, 'fuente': 'agroindustria', 'tiempo_segundos': 0.63}
{'escenario': 1, 'fuente': 'costos', 'tiempo_segundos': 0.54}
{'escenario': 1, 'fuente': 'abastecimiento', 'tiempo_segundos': 0.55}
{'escenario': 1, 'fuente': 'mercado_productores', 'tiempo_segundos': 0.76}
{'escenario': 10, 'fuente': 'produccion', 'tiempo_segundos': 1.39}
{'escenario': 10, 'fuente': 'exportaciones/regionales', 'tiempo_segundos': 0.98}
{'escenario': 10, 'fuente': 'exportaciones/danper', 'tiempo_segundos': 0.59}
{'escenario': 10, 'fuente': 'agroindustria', 'tiempo_segundos': 0.55}
{'escenario': 1